# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

#- **Unit of analysis:** One row = one content item (`content_hash_id`) for one client
#  (`client_hash_id`), aggregated over one month of daily performance.
#- **Tables:** `dim_content` (content metadata), `dim_clients` (for `gsc_data_start` /
#  `ga4_data_start` checks), `fact_content_daily_performance` (daily facts, partition
#  `month=2026-03`).
#- **Time window:** a mid-panel month, `report_date` between `2026-03-01` and `2026-03-31`
#  — deliberately not the final month, since that is the sealed test window.
#- **What I'd predict/rank:** a proxy target for CTR/Engagement Opportunity Scoring — content
#  items with high impressions but CTR below what's expected for their position tier (a CTR
#  gap), ranked as candidates for title/meta review. This is scoring/ranking, not necessarily
#  a binary supervised label this week.
#- **What I deliberately exclude:** product-decision flags such as `health_score`,
#  `priority_score`, `action_type` — these don't exist in this release, but would be excluded
#  as outputs of an existing decision, never as features. Raw query/URL/title fields are also
#  excluded — they are never part of this release in the first place.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
%pip -q install duckdb huggingface_hub
import os, getpass
import duckdb
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

#- **Feature** (knowable before the decision point): `impressions_month`, `clicks_month`,
#  `avg_position_month`, `days_with_data`, `content_age_days`.
#- **Label / proxy** (the thing being predicted, never a feature): `low_ctr_flag`, derived from
#  `ctr_month`.
#- **Context** (grouping/joining only, never model input): `client_hash_id`, `content_hash_id`.
#- **Excluded:**
#  - `health_score`, `priority_score`, `action_type` — not present in this release, but would
#    be excluded as product-decision outputs, not observable evidence.
#  - raw query/URL/title fields — never shipped in this release.
#  - `ctr_month` as a feature — excluded from the feature set once it's used to define the
#    label, because a feature that IS the label in disguise is circular (see the leakage trap
#    below). It is used only to build the proxy label.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# Three checks: grain, row count/date span, and availability — each with real query output.

grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MONTH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Grain violations found:", len(grain_check))
grain_check

span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_MONTH}
    WHERE gsc_impressions > 0
""").df()
span

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {FACT_MONTH}
""").df()
availability

### Five features — built from the same month

#Each feature gets a one-line "available when?" note.

#- `impressions_month` — available when: known throughout the month as GSC collects data; does
#  not depend on any future window.
#- `clicks_month` — available when: same, aggregated within the same month.
#- `avg_position_month` — available when: monthly average position, known at month's end.
#- `days_with_data` — available when: counts days with data inside the month, known once the
#  month closes.
#- `content_age_days` — available when: `content_created_at` is known from the moment the
#  content was created, always before any decision point.

features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_month,
        SUM(f.gsc_clicks) AS clicks_month,
        AVG(f.gsc_avg_position) AS avg_position_month,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) * 100 AS ctr_month,
        COUNT(DISTINCT f.report_date) AS days_with_data
    FROM {FACT_MONTH} f
    WHERE f.gsc_impressions > 0
    GROUP BY 1, 2
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(len(features), "content items")
features.head()

content_age = con.sql(f"""
    SELECT content_hash_id,
           DATE_DIFF('day', content_created_date, DATE '2026-03-31') AS content_age_days
    FROM {DIM_CONTENT}
""").df()

features = features.merge(content_age, on="content_hash_id", how="left")
features.head()

### The trap — deliberate leak

#`ctr_month` is used to define the proxy label `low_ctr_flag`. If I also use `ctr_month` as a
#feature, the "model" isn't learning anything — it's just reading the label back. Watch the
#score jump toward perfect, then the feature gets removed.

# Define the proxy label
features["low_ctr_flag"] = (features["ctr_month"] < 0.5).astype(int)

def precision_at_k(score, label, k):
    order = np.argsort(-np.asarray(score))
    return np.asarray(label)[order[:k]].mean()

# Honest score — has nothing to do with the label
honest_score = features["impressions_month"]
print("Honest precision@20:", precision_at_k(honest_score, features["low_ctr_flag"], 20))

# TRAP: add a feature that IS the label in disguise
features["leaky_feature"] = features["ctr_month"]
print("Leaky precision@20:", precision_at_k(-features["leaky_feature"], features["low_ctr_flag"], 20))
# -> near 1.0, because leaky_feature is literally the column the label was built from

# Remove the leak
features = features.drop(columns=["leaky_feature"])
print("Leak feature removed. Keeping only honest features.")

Grain violations found: 0
101441 content items
Honest precision@20: 0.7
Leaky precision@20: 1.0
Leak feature removed. Keeping only honest features.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

**Limitation — unbalanced panel:** Client history depth varies a lot (`gsc_data_start` /
`ga4_data_start` differ per client), and rows before a client's `ga4_data_start` have GA4
columns zero-filled with `ga4_data_available = FALSE`. Because of this, CTR/engagement results
from March 2026 alone may not generalize evenly to clients who joined later or have short
history — this single-month slice doesn't capture seasonality or newer clients' behavior.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.